In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


# Create and run a local RAG pipeline

## Goals for RAG

* Retrieval - Find relevant info given a query, e.g. "what are the macronutrients and what do they do?" -> retrieves passages of text realted to the macronutrients from a nutrition textbok.
* Augmented - Take the relevant info and augment our input(prompt) to an LLM with that relevant info
* Generation - Take the first two steps and pass them to an LLM for generative outputs

## Main Goal
improve the outputs of LLMs by:
1. Preventing hallucinations
2. Work with custom data


## Use Cases:
* Textbook Q&A - find releveant passages to the questions you have.

In [9]:
import os
import requests

# Get PDF document path
pdf_path = "sustainability-action-plan.pdf"

# Download PDF
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading...")

    #Enter the URL of the PDF
    url = "https://cityofwinterpark.org/docs/departments/natural-resources-sustainability/sustainability-division/sustainability-action-plan/sustainability-action-plan-annual-report-2023.pdf"

    #The local filename to save the downloaded file
    filename=pdf_path

    # Browser-like User-Agent required: the site returns 403 for python-requests' default
    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}


    #Send a GET request to the URL
    response = requests.get(url, headers=headers, timeout=30)

    #Check if the request was successful
    if response.status_code ==200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)

        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[INFO] Failed to download the file. Status code: {response.status_code}")

else:
    print(f"File {pdf_path} exists.")



File sustainability-action-plan.pdf exists.


In [11]:
pip install tqdm

In [24]:
import pymupdf # requires: !pip install PyMuPDF
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text= text.replace("\n"," ").strip()

    return cleaned_text

PRINTED_PAGE_OFFSET = 2
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    doc = pymupdf.open(pdf_path)
    pages_and_texts=[]
    
    for page_number, page in tqdm(enumerate(doc)):
        if page_number < PRINTED_PAGE_OFFSET:
            continue # skip the two pages before the title page
        
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({"page_number": page_number - PRINTED_PAGE_OFFSET + 1,
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count": len(text.split(". ")),
                                "page_token_count": len(text) / 4, # 1 token = ~ 4 characters
                                "text": text})

    return pages_and_texts
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': 1,
  'page_char_count': 406,
  'page_word_count': 114,
  'page_sentence_count': 1,
  'page_token_count': 101.5,
  'text': 'City of Winter Park  Sustainability Action Plan                  January 24th, 2024                            Presented by:   Gloria Eby, Natural Resources and Sustainability Director     Sara Miller, Sustainability Manager   Mia Brady, Sustainability Specialist   Keep Winter Park Beautiful & Sustainable Advisory Board           Certified WCAG 2.1 Level AA compliant and usable with JAWS and NVDA by SWH'},
 {'page_number': 2,
  'page_char_count': 2873,
  'page_word_count': 155,
  'page_sentence_count': 22,
  'page_token_count': 718.25,
  'text': '2    Contents   Background ........................................................................................................................................... 3  Vision and Purpose ......................................................................................................................

In [25]:
import random
random.sample(pages_and_texts, k=3)

[{'page_number': 43,
  'page_char_count': 2202,
  'page_word_count': 393,
  'page_sentence_count': 16,
  'page_token_count': 550.5,
  'text': '43  Waste Management  The Waste Management category is focused on reducing  the amount of waste generated, encouraging the reuse  and repair of products, and diverting waste from the  landfill. The EPA developed the non-hazardous materials  and waste management hierarchy in recognition that no  single waste management approach is suitable for  managing all materials and waste streams in all  circumstances.  The  hierarchy  ranks  the  various  management  strategies  from  most  to  least  environmentally preferred.    The City of Winter Park has a franchise agreement with WastePro for hauling solid waste,  yard waste, and recyclables from residential properties.  Under this contract, WastePro  hauls solid waste to the Seminole County Transfer Station with exceptions during diversion  events. Solid waste is landfilled, and yard  waste is primari

In [26]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count,text
0,1,406,114,1,101.50,City of Winter Park Sustainability Action Pla...
1,2,2873,155,22,718.25,2 Contents Background ......................
2,3,2448,420,12,612.00,3 Background Vision and Purpose The 2022 Su...
3,4,2954,505,16,738.50,"4 In 2012, the city’s Environmental Review an..."
4,5,2551,411,7,637.75,5 indicators to benchmark city status in effo...


In [27]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count
count,57.0,57.00,57.00,57.00,57.00
mean,29.0,2050.11,353.60,9.28,512.53
std,16.6,701.39,113.62,7.93,175.35
min,1.0,406.00,114.00,1.00,101.50
25%,15.0,1559.00,279.00,2.00,389.75
50%,29.0,1958.00,366.00,7.00,489.50
75%,43.0,2420.00,420.00,15.00,605.00
max,57.0,3876.00,639.00,33.00,969.00
